In [ ]:
!pip install transformers datasets sentencepiece -q

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k")
print(dataset)

In [ ]:
small_dataset = dataset["train"].select(range(5))

In [ ]:
print(small_dataset[0])

In [ ]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def translate_text_en_to_mk(text):
    tokenizer.src_lang = "eng_Latn"   # source language

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("mkd_Cyrl"),
        max_length=512
    )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

In [ ]:
translated_examples = []

for ex in small_dataset:
    original_instruction = ex["instruction"]
    original_response = ex["response"]
    original_context = ex["context"]

    translated_instruction = translate_text_en_to_mk(original_instruction)
    translated_response = translate_text_en_to_mk(original_response)

    if original_context:
        translated_context = translate_text_en_to_mk(original_context)
    else:
        translated_context = ""

    translated_examples.append({
        "mk_instruction": translated_instruction,
        "mk_response": translated_response,
        "mk_context": translated_context
    })

In [ ]:
for ex in translated_examples:
    print("TRANSLATED INSTRUCTION:")
    print(ex["mk_instruction"])

    print("\nTRANSLATED RESPONSE:")
    print(ex["mk_response"])

    print("\n" + "="*80 + "\n")

In [ ]:
import pandas as pd
df = pd.DataFrame(translated_examples)

df
df = pd.DataFrame(translated_examples)[
    ["orig_instruction", "mk_instruction", "orig_response", "mk_response"]
]

df
pd.set_option("display.max_colwidth", None)
df
df.reset_index(inplace=True)
df.rename(columns={"index": "ID"}, inplace=True)
df